In [ ]:
import pandas as pd
import numpy as np
import igraph as ip
import networkx as nx
import matplotlib.pyplot as plt
from fpdf import FPDF
import seaborn as sns
from scipy import stats
import os

In [ ]:
datiCelltypeRaw=pd.read_csv("../data/science.add9330_data_s2.csv", header=0)

# Tabella informazioni

In [ ]:
# Creo un dizionario dove mi salvo i neuroni per area di appartenenza, lo useremo per scansionare i nodi del
# grafo caricato prima e assegnare loro l'area di appartenenza
dizionario = {}

for index, row in datiCelltypeRaw.iterrows():
    # Ottenere il valore della colonna 'level_7_cluster' per la chiave del dizionario
    celltype_value = row['celltype']
    
    # Ottenere i valori delle colonne 'left_id' e 'right_id'
    left_id = row['left_id']
    right_id = row['right_id']
    
    # Verificare quale valore utilizzare, considerando "no pair"
    if left_id != "no pair" and right_id != "no pair":
        # Entrambi i valori sono validi, quindi aggiungi entrambi alla lista
        values = [left_id, right_id]
    elif left_id != "no pair":
        # Solo left_id è valido
        values = [left_id]
    elif right_id != "no pair":
        # Solo right_id è valido
        values = [right_id]
    else:
        # Entrambi i valori sono "no pair", quindi non aggiungere nulla
        values = []

    # Verificare se il valore del cluster è già presente nel dizionario
    if celltype_value in dizionario:
        # Aggiungere i valori a questa chiave esistente
        dizionario[celltype_value].extend(values)
    else:
        # Creare una nuova chiave nel dizionario e aggiungere i valori
        dizionario[celltype_value] = values

In [ ]:
#generazione del grafo
connectivity_matrix_df = pd.read_csv("../data/all-all_connectivity_matrix.csv", header=[0], index_col=[0])

# Estrai gli indici dei nodi e i valori della matrice di adiacenza
node_names = connectivity_matrix_df.columns.values
adj_matrix = connectivity_matrix_df.values #Peso delle connessioni
G = nx.DiGraph()
G.add_nodes_from(node_names)

# Aggiungi gli archi al grafo basati sulla matrice di adiacenza
for i in range(len(node_names)):
    for j in range(len(node_names)):
        if adj_matrix[i, j] != 0:
            G.add_edge(node_names[i], node_names[j])

In [ ]:
len(node_names)

In [ ]:
celltype_subgraphs = {}

for celltype in dizionario.keys():
    nodi = dizionario[celltype]
    sottografo = G.subgraph(nodi)
    celltype_subgraphs[celltype] = sottografo

In [ ]:
tabella = {}

for celltype in dizionario.keys():
    nodi = dizionario[celltype]
    celltype_info = {}

    sottografo = celltype_subgraphs[celltype]

    celltype_info['Number of nodes (neurons)'] = sottografo.number_of_nodes()
    celltype_info['Number of arcs (connections)'] = sottografo.number_of_edges()
    celltype_info['Density'] = nx.density(sottografo)
    celltype_info['Average Clustering Coefficient'] = nx.average_clustering(sottografo)
    celltype_info['Degree Assortativity'] = nx.degree_assortativity_coefficient(sottografo)
    sottografo = sottografo.to_undirected()
    celltype_info['Number of Connected Components'] = nx.number_connected_components(sottografo)
    Gcc = sorted(nx.connected_components(sottografo), key=len, reverse=True)
    G0 = sottografo.subgraph(Gcc[0])
    celltype_info['Max Connected Component'] = G0.number_of_nodes()
    
    
    tabella[celltype] = celltype_info

In [ ]:
for key, sub_dict in tabella.items():
    print(f"{key}:")
    for sub_key, value in sub_dict.items():
        print(f"  {sub_key}: {value}")

In [ ]:
tabella_df = pd.DataFrame.from_dict(tabella, orient="index")

tabella_df

# Distribuzioni centralità

In [ ]:
def plot_centrality_distributions(celltype, graph):
    centrality_types = ["Betweenness", "Closeness", "Degree", "Eigenvector"]
    centrality_functions = {
        "Betweenness": nx.betweenness_centrality,
        "Closeness": nx.closeness_centrality,
        "Degree": nx.degree_centrality,
        "Eigenvector": lambda g: nx.eigenvector_centrality(g, max_iter=2000)
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))  # Crea una griglia 2x2 per le sottofigure
    axes = axes.flatten()  # Appiattisce la griglia per iterare facilmente
    
    for ax, centrality_type in zip(axes, centrality_types):
        centrality = centrality_functions[centrality_type](graph)
        n = len(list(centrality.values()))
        bin_width = 3.5 * np.std(list(centrality.values())) / (n ** (1 / 3))
        centrality_values = list(centrality.values())
        mean_val = np.mean(centrality_values)
        median_val = np.median(centrality_values)
        
        sns.histplot(centrality_values, kde=True, label='Distribution', binwidth=bin_width, ax=ax)
        ax.axvline(mean_val, color='r', linestyle='--', label='Average')
        ax.axvline(median_val, color='g', linestyle='-', label='Median')
        ax.set_xticks(ax.get_xticks())
        ax.tick_params(axis='x', rotation=80, labelsize=8)
        ax.set_ylabel("Number of neurons", fontsize=10)
        ax.set_xlabel(centrality_type + " Centrality value", fontsize=10)
        ax.set_title(centrality_type + " Centrality distribution", fontsize=12)
        ax.legend(fontsize=8)
    
    plt.suptitle(f"Centrality Distributions - {celltype}", fontsize=16)  # Titolo complessivo
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adatta il layout evitando sovrapposizioni
    plt.savefig(f"Immagini Paper/centrality_distributions_{celltype}.pdf")
    plt.close()

In [ ]:
for celltype in dizionario.keys():
    sottografo = celltype_subgraphs[celltype]
    plot_centrality_distributions(celltype, sottografo)
    

# Scatterplot centralità

In [ ]:
def plot_centrality_pairs(celltype, graph):
    # Definizione delle centralità
    centrality_types = ["Betweenness", "Closeness", "Degree", "Eigenvector"]
    centrality_functions = {
        "Betweenness": nx.betweenness_centrality,
        "Closeness": nx.closeness_centrality,
        "Degree": nx.degree_centrality,
        "Eigenvector": lambda g: nx.eigenvector_centrality(g, max_iter=2000)
    }
    
    # Calcolo delle centralità
    centralities = {
        ctype: centrality_functions[ctype](graph) for ctype in centrality_types
    }
    
    # Generazione delle coppie di centralità
    pairs = list(itertools.combinations(centrality_types, 2))
    
    # Calcolo delle dimensioni della griglia
    n_pairs = len(pairs)
    n_cols = 2  # Numero di colonne
    n_rows = (n_pairs + n_cols - 1) // n_cols  # Calcolo righe necessarie
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
    axes = axes.flatten()  # Appiattisce la griglia
    
    for ax, (ctype1, ctype2) in zip(axes, pairs):
        # Ottenere i valori delle centralità
        values1 = list(centralities[ctype1].values())
        values2 = list(centralities[ctype2].values())
        
        # Creazione del grafico
        sns.scatterplot(x=values1, y=values2, ax=ax, s=100)
        ax.set_xlabel(f"{ctype1} Centrality", fontsize=10)
        ax.set_ylabel(f"{ctype2} Centrality", fontsize=10)
        ax.set_title(f"{ctype1} vs {ctype2}", fontsize=12)
    
    # Rimuovere subplot vuoti se il numero di coppie è dispari
    for i in range(len(pairs), len(axes)):
        fig.delaxes(axes[i])
    
    plt.suptitle(f"Centrality Pairs - {celltype}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f"Immagini Paper/centrality_pairs_{celltype}.pdf")
    plt.close()


In [ ]:
for celltype in dizionario.keys():
    sottografo = celltype_subgraphs[celltype]
    plot_centrality_pairs(celltype, sottografo)

# Triad census

In [ ]:
for celltype in dizionario.keys():
    sottografo = celltype_subgraphs[celltype]
    triadic_census = nx.triadic_census(sottografo)
    print('Triad Census ', celltype)
    for key, value in triadic_census.items():
        print(f"{key}: {value}")

# Boxplot centralità

In [ ]:
def plot_centrality_boxplots_across_celltypes(celltypes, graphs, centrality_type):
    """
    Crea boxplot per una specifica centralità (ad esempio 'Betweenness') per diverse celltype.

    Args:
        celltypes: Lista dei nomi delle celltype.
        graphs: Lista dei grafi corrispondenti alle celltype.
        centrality_type: Tipo di centralità da calcolare (Betweenness, Closeness, Degree, Eigenvector).
    """
    # Associare le funzioni di calcolo delle centralità
    centrality_functions = {
        "Betweenness": nx.betweenness_centrality,
        "Closeness": nx.closeness_centrality,
        "Degree": nx.degree_centrality,
        "Eigenvector": lambda g: nx.eigenvector_centrality(g, max_iter=2000)
    }
    
    # Verificare che la centralità selezionata sia valida
    if centrality_type not in centrality_functions:
        raise ValueError(f"Centrality '{centrality_type}' not recognized. Choose from {list(centrality_functions.keys())}.")
    
    # Calcolare i valori di centralità per ciascun grafo
    data = []
    for celltype, graph in zip(celltypes, graphs):
        centrality_values = list(centrality_functions[centrality_type](graph).values())
        data.append((celltype, centrality_values))
    
    # Preparare i dati per il boxplot
    plot_data = []
    labels = []
    for celltype, values in data:
        plot_data.append(values)
        labels.append(celltype)
    
    # Configurazione del grafico
    plt.figure(figsize=(12, 9))
    sns.boxplot(data=plot_data, width=0.6)
    
    # Impostazioni degli assi e delle etichette
    plt.xticks(ticks=range(len(labels)), labels=labels, fontsize=11, rotation=45)
    plt.ylabel(f"{centrality_type} Centrality Value", fontsize=12)
    plt.title(f"{centrality_type} Centrality Across Celltypes", fontsize=14)
    
    # Salvataggio e chiusura
    plt.savefig(f"Immagini Paper/{centrality_type.lower()}_centrality_across_celltypes.pdf")
    plt.close()


In [ ]:
celltypes = list(dizionario.keys())
graphs = celltype_subgraphs.values()

plot_centrality_boxplots_across_celltypes(celltypes, graphs, centrality_type="Degree")
plot_centrality_boxplots_across_celltypes(celltypes, graphs, centrality_type="Closeness")
plot_centrality_boxplots_across_celltypes(celltypes, graphs, centrality_type="Betweenness")
plot_centrality_boxplots_across_celltypes(celltypes, graphs, centrality_type="Eigenvector")

# Distanza fisica tra emisferi

In [ ]:
coordinate_neuroni = pd.read_csv('Coordinate V.1.csv')
coordinate_neuroni = coordinate_neuroni.drop('skeleton_id', axis=1)

In [ ]:
matrice_adiacenza = pd.read_csv("../data/all-all_connectivity_matrix.csv", header=[0], index_col=[0])
matrice_adiacenza = matrice_adiacenza.map(lambda x: 1 if x != 0 else 0)

In [ ]:
coordinate_dict = coordinate_neuroni.set_index('id')[['x', 'y', 'z']].to_dict(orient='index')

In [ ]:
coordinate_dict

In [ ]:
# Funzione per calcolare la distanza euclidea tra due nodi
def distanza_euclidea(id1, id2):
    coord1 = coordinate_dict[id1]
    coord2 = coordinate_dict[id2]
    return np.sqrt((coord2['x'] - coord1['x'])**2 + (coord2['y'] - coord1['y'])**2 + (coord2['z'] - coord1['z'])**2)

In [ ]:
# Iteriamo sulla matrice di adiacenza e sostituiamo l'1 con la distanza euclidea
for i in range(matrice_adiacenza.shape[0]):
    for j in range(matrice_adiacenza.shape[1]):
        id1 = matrice_adiacenza.index[i]  # id del nodo dalla riga
        id2 = int(matrice_adiacenza.columns[j])  # id del nodo dalla colonna
        distanza = distanza_euclidea(id1, id2)
        # Sostituisci 0 o 1 con la distanza
        matrice_adiacenza.iloc[i, j] = int(distanza)

In [ ]:
# Funzione per calcolare la distanza euclidea tra due nodi
def distanza_euclidea(coord1, coord2):
    return np.linalg.norm(np.array(coord1) - np.array(coord2))

# Creiamo una matrice delle distanze precalcolata tra tutti i nodi
def calcola_matrice_distanze(coordinate_dict):
    # Estrai le coordinate di tutti i nodi
    nodi = list(coordinate_dict.keys())
    n = len(nodi)

    # Pre-alloca una matrice delle distanze
    matrice_distanze = np.zeros((n, n))

    # Calcola la distanza tra tutti i nodi
    for i in range(n):
        for j in range(n):  # calcola solo la metà superiore della matrice (simmetrica)
            id1, id2 = nodi[i], nodi[j]
            distanza = distanza_euclidea(coordinate_dict[id1], coordinate_dict[id2])
            matrice_distanze[i, j] = distanza
            matrice_distanze[j, i] = distanza  # matrice simmetrica

    return matrice_distanze

# Usa la matrice delle distanze per aggiornare la matrice di adiacenza
def aggiorna_matrice_adiacenza(matrice_adiacenza, matrice_distanze, nodi):
    # Utilizziamo il mapping degli id dei nodi per aggiornare la matrice di adiacenza
    for i, id1 in enumerate(nodi):
        for j, id2 in enumerate(nodi):
            if matrice_adiacenza.iloc[i, j] == 1:  # Cambia solo dove c'è un 1 (connessione)
                matrice_adiacenz

In [ ]:
# Funzione per calcolare la distanza euclidea tra due nodi in 3D
def distanza_euclidea(id1, id2, coordinate_dict):
    coord1 = coordinate_dict[id1]  # coordinate del nodo id1
    coord2 = coordinate_dict[id2]  # coordinate del nodo id2
    # Calcola la distanza euclidea in 3D
    return np.sqrt((coord1['x'] - coord2['x'])**2 + 
                   (coord1['y'] - coord2['y'])**2 + 
                   (coord1['z'] - coord2['z'])**2)

# Funzione per calcolare la matrice delle distanze
def calcola_matrice_distanze(coordinate_dict):
    nodi = list(coordinate_dict.keys())  # Lista di tutti gli id dei nodi
    n = len(nodi)

    # Pre-alloca una matrice delle distanze (matrice quadrata n x n)
    matrice_distanze = np.zeros((n, n))

    # Calcola la distanza tra tutti i nodi
    for i in range(n):
        if i%100==0:
                print(i)
        for j in range(i + 1, n): # calcola solo la metà superiore della matrice (simmetrica)
            id1, id2 = nodi[i], nodi[j]
            distanza = distanza_euclidea(id1, id2, coordinate_dict)
            matrice_distanze[i, j] = distanza
            matrice_distanze[j, i] = distanza  # matrice simmetrica
    return matrice_distanze

# Funzione per aggiornare la matrice di adiacenza
def aggiorna_matrice_adiacenza(matrice_adiacenza, matrice_distanze, nodi):
    # Utilizziamo il mapping degli id dei nodi per aggiornare la matrice di adiacenza
    for i, id1 in enumerate(nodi):
        if i%100==0:
                print(i)
        for j, id2 in enumerate(nodi):
            matrice_adiacenza.iloc[i, j] = matrice_distanze[i, j]
            matrice_adiacenza.iloc[j, i] = matrice_distanze[i, j]
    return matrice_adiacenza


In [ ]:
# Calcoliamo la matrice delle distanze
matrice_distanze = calcola_matrice_distanze(coordinate_dict)

# Aggiorniamo la matrice di adiacenza con le distanze
nodi = matrice_adiacenza.index.to_list()
matrice_adiacenza = aggiorna_matrice_adiacenza(matrice_adiacenza, matrice_distanze, nodi)


In [ ]:
matrice_adiacenza

In [ ]:
coordinate_dict[11717035]

In [ ]:
coordinate_dict[29]